# 🧩 Aplicaciones con Transformers

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 13 · Bloque 3 — 50 minutos**

---

## Objetivo

Usar pipelines de Hugging Face para resolver tareas reales de PLN con modelos preentrenados listos para producción.

## Al terminar este bloque vas a poder:

1. Implementar análisis de sentimiento y clasificación zero-shot en español.
2. Aplicar sumarización automática y traducción neural.
3. Elegir el tipo de modelo adecuado para cada tarea.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Pipeline** | Objeto de alto nivel que encapsula tokenización + inferencia + postprocesamiento en una sola llamada. |
| **Zero-Shot Classification** | Capacidad de clasificar texto en categorías sin haber sido entrenado explícitamente en ellas. |
| **NER (Named Entity Recognition)** | Tarea que detecta y clasifica entidades nombradas en el texto: personas, lugares, organizaciones. |
| **Análisis de sentimiento** | Clasificar el tono emocional de un texto: positivo, negativo o neutral. |
| **Sumarización abstractiva** | Generar un resumen con oraciones nuevas (no copiadas del texto original). |

## ⚙️ Instalación

Este bloque necesita `transformers` y dos librerías que usan los tokenizadores **SentencePiece** de mT5 (sumarización) y MarianMT (traducción).

> ⚠️ **Importante:** después de instalar `sentencepiece`, **reiniciá el kernel** (`Kernel → Restart`) antes de seguir. Si no, las Partes 3 y 4 van a fallar.
>
> - **En Colab:** corré la celda de abajo tal cual.
> - **En local con uv:** corré desde la terminal `uv pip install transformers sentencepiece protobuf` y reiniciá el kernel.

In [ ]:
# En Colab funciona directo. En local con uv, ver la nota de arriba.
!pip install -q transformers sentencepiece protobuf

## ¿Qué es un pipeline?

### Analogía

Un pipeline es como un electrodoméstico ya armado: vos ponés los ingredientes (el texto), apretás un botón (llamás la función), y sale el resultado listo para usar. No necesitás saber cómo funciona el motor adentro — el pipeline maneja la tokenización, la inferencia y la decodificación por vos.

### Dónde vive esto en el mundo real

Los pipelines de Hugging Face son el puente entre los modelos académicos y las aplicaciones reales. Empresas como Mercado Libre, Globant y Santander los usan para análisis de reviews, clasificación de tickets de soporte y detección de fraude en textos. En este bloque vas a resolver **cuatro tareas** distintas con el mismo patrón de código.

### Arquitecturas según la tarea

| Tarea | Arquitectura | Ejemplos |
|---|---|---|
| Clasificación, análisis | **Encoder** (BERT) | Sentimiento, NER, zero-shot |
| Generación de texto | **Decoder** (GPT) | Completar prompts, chatbots |
| Traducción, sumarización | **Encoder-Decoder** (T5) | Transformar un texto en otro |

### ✎ Para pensar

- ¿Por qué tiene sentido usar un modelo Encoder para clasificar y un Decoder para generar texto?
- ¿Qué desventaja tiene usar un modelo multilingüe genérico frente a uno entrenado específicamente en español?

## Parte 1 — Análisis de sentimiento

Usamos `finiteautomata/beto-sentiment-analysis`, un modelo basado en BETO entrenado con tweets en español rioplatense. Clasifica en **POS / NEG / NEU** y devuelve un score de confianza.

In [ ]:
from transformers import pipeline

# Cargamos el pipeline de análisis de sentimiento
# model: Especifica qué modelo usar (finiteautomata/beto-sentiment-analysis está entrenado con tweets argentinos)
sentiment = pipeline(
    "sentiment-analysis",
    model="finiteautomata/beto-sentiment-analysis"
)

print("Pipeline de sentimiento cargado exitosamente")

In [ ]:
# Frases de prueba con diferentes sentimientos
frases = [
    "Este lugar está buenísimo, la atención de diez",
    "Una experiencia horrible, me quiero ir",
    "Zafa, pero esperaba más",
    "Recomiendo totalmente este producto",
    "Nunca más compro acá",
    "El asado estaba en su punto, espectacular",
    "Meh, ni fu ni fa"
]

print("Análisis de Sentimiento:")
print("=" * 80)

for frase in frases:
    resultado = sentiment(frase)[0]  # El pipeline devuelve una lista, tomamos el primer elemento
    etiqueta = resultado['label']     # Etiqueta: POS, NEG, NEU
    confianza = resultado['score']    # Score de confianza (0-1)

    print(f"\nTexto: '{frase}'")
    print(f"Sentimiento: {etiqueta} (confianza: {confianza:.3f})")

### Interpretando el score

| Score | Interpretación |
|---|---|
| > 0.90 | Alta confianza — sentimiento claro |
| 0.70 – 0.90 | Confianza moderada |
| < 0.70 | Baja confianza — texto ambiguo o irónico |

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Probá el modelo con frases de tu área de interés:
#   1. Frases con ironía o sarcasmo — ¿el modelo las detecta bien?
#   2. Frases cortas vs largas — ¿varía la confianza?
#   3. Frases en lunfardo o muy coloquiales
#
# ─────────────────────────────────────────────────────────────────────────────
mis_frases = [
    "Tu frase 1",
    "Tu frase 2",
]
for frase in mis_frases:
    r = sentiment(frase)[0]
    print(f"{r['label']:4} ({r['score']:.2f}) | {frase}")

### ✎ Para pensar

- Probaste frases con sarcasmo. ¿El modelo las clasificó bien? ¿Por qué es difícil detectar sarcasmo automáticamente?
- El modelo fue entrenado con tweets argentinos. ¿Esperarías resultados igual de buenos con tweets chilenos o españoles?

## Parte 2 — Clasificación zero-shot

El modelo no fue entrenado con tus categorías — las infiere por comprensión del lenguaje. Internamente, convierte cada etiqueta en una hipótesis y evalúa qué tan compatible es con el texto (tarea de NLI: Natural Language Inference).

In [ ]:
# Cargamos el pipeline de clasificación zero-shot
# Este modelo fue entrenado con el dataset XNLI (cross-lingual natural language inference)
classifier = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli"
)

print("Pipeline de clasificación zero-shot cargado")

In [ ]:
# Texto a clasificar
texto = "Lionel Messi firmó contrato con el Inter Miami y debutará esta semana en la MLS."

# Categorías candidatas (podés usar las categorías que quieras)
etiquetas = ["deportes", "economía", "política", "espectáculos", "tecnología"]

# Clasificamos
# candidate_labels: Lista de posibles categorías
resultado = classifier(texto, candidate_labels=etiquetas)

print(f"Texto: '{texto}'\n")
print("Clasificación:")
print("=" * 60)

# El resultado incluye las etiquetas ordenadas por score
for etiqueta, score in zip(resultado['labels'], resultado['scores']):
    print(f"{etiqueta:15} | {score:.4f} ({score*100:.2f}%)")

In [ ]:
# Probemos con más ejemplos
textos_prueba = [
    "El Congreso aprobó la nueva ley de presupuesto después de intensas negociaciones entre los bloques.",
    "Apple presentó su nuevo iPhone con pantalla plegable y cámara de 200 megapíxeles.",
    "La obra de teatro recibió tres premios en el festival internacional de artes escénicas.",
    "El Banco Central decidió mantener la tasa de interés en 75% para controlar la inflación."
]

for texto in textos_prueba:
    resultado = classifier(texto, candidate_labels=etiquetas)
    categoria_principal = resultado['labels'][0]
    confianza = resultado['scores'][0]

    print(f"\nTexto: '{texto[:70]}...'")
    print(f"Categoría: {categoria_principal} (confianza: {confianza:.3f})")

### ¿Cuándo usar zero-shot?

| Situación | Recomendación |
|---|---|
| No tenés datos etiquetados | ✅ Zero-shot |
| Las categorías cambian seguido | ✅ Zero-shot |
| Necesitás alta precisión (>90%) | ⚠️ Mejor entrenar un clasificador |
| Tenés cientos de ejemplos etiquetados | ⚠️ Mejor hacer fine-tuning |

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Clasificá tus propios textos cambiando las categorías:
#   1. Elegí un texto (una noticia, un tweet, una reseña)
#   2. Definí 3-5 categorías candidatas
#   3. Observá cómo cambia el resultado si reformulás las etiquetas
#      (ej: 'deportes' vs 'noticias deportivas')
#
# ─────────────────────────────────────────────────────────────────────────────
mi_texto = "Tu texto aquí"
mis_categorias = ["categoría 1", "categoría 2", "categoría 3"]

resultado = classifier(mi_texto, candidate_labels=mis_categorias)
print(f"Categoría: {resultado['labels'][0]} ({resultado['scores'][0]:.3f})")

### ✎ Para pensar

- El modelo nunca vio tus categorías durante el entrenamiento. ¿Cómo hace entonces para clasificar?
- ¿Por qué reformular una etiqueta (ej: 'política' → 'noticias de política') puede cambiar el resultado?

## Parte 3 — Sumarización automática

Usamos `mT5_multilingual_XLSum`: un modelo encoder-decoder entrenado para generar resúmenes de una sola oración en más de 40 idiomas. El resumen es **abstractivo** — genera oraciones nuevas, no copia fragmentos del original.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Cargamos modelo y tokenizador de sumarización directamente
modelo_id = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer_sum = AutoTokenizer.from_pretrained(modelo_id)
modelo_sum = AutoModelForSeq2SeqLM.from_pretrained(modelo_id)

# Función auxiliar que reemplaza al pipeline
def summarizer(texto, max_length=50, min_length=20, **kwargs):
    # 1. Convertimos el texto en tokens
    entradas = tokenizer_sum(texto, return_tensors="pt", truncation=True)

    # 2. Generamos el resumen
    ids_resumen = modelo_sum.generate(
        **entradas,
        max_length=max_length,
        min_length=min_length,
        no_repeat_ngram_size=2
    )

    # 3. Decodificamos los tokens de vuelta a texto
    texto_resumen = tokenizer_sum.decode(ids_resumen[0], skip_special_tokens=True)

    # Devolvemos en el mismo formato que usaba el pipeline
    return [{"summary_text": texto_resumen}]

print("Pipeline de sumarización cargado")

In [ ]:
# Texto de ejemplo (noticia)
texto_largo = """
El Ministerio de Salud confirmó hoy que se logró una reducción sostenida de casos de dengue
en las últimas semanas. Según los datos provistos, hubo una disminución del 40% en comparación
con el pico de marzo. Las campañas de prevención, sumadas a la llegada del frío, contribuyeron
a esta baja. Sin embargo, las autoridades sanitarias piden a la población mantener las precauciones,
especialmente eliminando recipientes con agua estancada donde el mosquito puede reproducirse.
El director de Epidemiología destacó que, aunque los números son alentadores, es fundamental no
bajar la guardia. Se espera que con el invierno los casos continúen descendiendo, pero la
vigilancia epidemiológica seguirá activa.
"""

# Generamos el resumen
# max_length: Longitud máxima del resumen en tokens
# min_length: Longitud mínima del resumen en tokens
# do_sample: False para generación determinística
resumen = summarizer(
    texto_largo,
    max_length=50,
    min_length=20,
    do_sample=False
)

print("Texto original:")
print(texto_largo.strip())
print("\n" + "="*80 + "\n")
print("Resumen generado:")
print(resumen[0]['summary_text'])

In [ ]:
# Probemos con otro texto
texto_economia = """
La inflación en Argentina mostró una leve desaceleración en el último mes, según el informe
del INDEC publicado esta mañana. El índice de precios al consumidor registró un aumento del
7.2%, por debajo del 8.4% del mes anterior. Los analistas atribuyen esta baja a la política
monetaria restrictiva implementada por el Banco Central y a la estabilización del tipo de
cambio. Sin embargo, advierten que la tendencia aún no se revirtió de manera definitiva, y
podrían esperarse nuevos aumentos en los próximos meses si no se sostienen las medidas actuales.
Los sectores más afectados continúan siendo alimentos y bebidas, con incrementos superiores al
promedio general.
"""

resumen_economia = summarizer(
    texto_economia,
    max_length=60,
    min_length=25,
    do_sample=False
)

print("Texto original:")
print(texto_economia.strip())
print("\n" + "="*80 + "\n")
print("Resumen:")
print(resumen_economia[0]['summary_text'])

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Resumí un texto propio:
#   1. Pegá una noticia o un artículo (3-4 párrafos)
#   2. Ajustá max_length y min_length según lo conciso que lo quieras
#   3. Compará el resumen con lo que vos resumirías a mano
#
# ─────────────────────────────────────────────────────────────────────────────
mi_texto = """
Pegá acá tu texto largo.
"""

mi_resumen = summarizer(mi_texto, max_length=50, min_length=20)
print(mi_resumen[0]['summary_text'])

### ✎ Para pensar

- El resumen es *abstractivo*: el modelo genera oraciones nuevas. ¿Qué riesgo tiene eso frente a uno *extractivo* que solo copia frases del original?
- Probá bajar `max_length` a 30. ¿El resumen sigue siendo coherente o se corta?

## Parte 4 — Traducción automática

`Helsinki-NLP/opus-mt-es-en` es un modelo MarianMT entrenado en el corpus OPUS (millones de pares de oraciones paralelas). Es específico para el par español → inglés.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Cargamos modelo y tokenizador de traducción español → inglés
modelo_id = "Helsinki-NLP/opus-mt-es-en"
tokenizer_tr = AutoTokenizer.from_pretrained(modelo_id)
modelo_tr = AutoModelForSeq2SeqLM.from_pretrained(modelo_id)

# Función auxiliar que reemplaza al pipeline
def translator(texto, **kwargs):
    # 1. Convertimos el texto en tokens
    entradas = tokenizer_tr(texto, return_tensors="pt", truncation=True)

    # 2. Generamos la traducción
    ids_traduccion = modelo_tr.generate(**entradas, max_length=512)

    # 3. Decodificamos los tokens de vuelta a texto
    texto_traducido = tokenizer_tr.decode(ids_traduccion[0], skip_special_tokens=True)

    # Devolvemos en el mismo formato que usaba el pipeline
    return [{"translation_text": texto_traducido}]

print("Pipeline de traducción cargado")

In [ ]:
# Textos para traducir
textos_espanol = [
    "La inteligencia artificial está cambiando el mundo.",
    "El mate es la bebida tradicional de Argentina y Uruguay.",
    "Los modelos de lenguaje pueden generar texto coherente y contextualmente apropiado.",
    "Me encanta programar en Python porque es simple y poderoso."
]

print("Traducción Español → Inglés")
print("=" * 80)

for texto in textos_espanol:
    traduccion = translator(texto)[0]['translation_text']
    print(f"\nES: {texto}")
    print(f"EN: {traduccion}")

In [ ]:
# Probemos con expresiones coloquiales argentinas
expresiones = [
    "Me colgué y se me hizo tarde.",
    "La película estuvo de diez, te la súper recomiendo.",
    "Aguante el mate y las facturas."
]

print("Traduciendo expresiones coloquiales:")
print("=" * 80)

for expresion in expresiones:
    traduccion = translator(expresion)[0]['translation_text']
    print(f"\nES: {expresion}")
    print(f"EN: {traduccion}")
    print("Nota: Observá si mantiene el sentido original o traduce literalmente")

In [ ]:
# ─── Espacio de práctica ──────────────────────────────────────────────────────
#
# Traducí tus propias frases:
#   1. Probá frases formales y frases muy coloquiales (lunfardo)
#   2. Observá cuáles mantiene bien y cuáles traduce literalmente
#
# ─────────────────────────────────────────────────────────────────────────────
mis_textos = [
    "Tu frase formal aquí",
    "Tu frase coloquial aquí",
]

for texto in mis_textos:
    traduccion = translator(texto)[0]['translation_text']
    print(f"ES: {texto}")
    print(f"EN: {traduccion}\n")

### ✎ Para pensar

- MarianMT es específico para el par español → inglés. ¿Qué ventaja tiene frente a un modelo que traduce 'cualquier idioma a cualquier idioma'?
- Las expresiones coloquiales (lunfardo) suelen traducirse mal. ¿Por qué pasa eso si el modelo vio millones de oraciones?

## ⛰️ Cierre del bloque

| Tarea | Modelo | Arquitectura |
|---|---|---|
| Análisis de sentimiento | BETO Sentiment | Encoder (BERT) |
| Clasificación zero-shot | BERT XNLI | Encoder (BERT) |
| Sumarización | mT5 XLSum | Encoder-Decoder |
| Traducción | MarianMT | Encoder-Decoder |

**Patrón común:** elegir el modelo según la tarea (Encoder para analizar, Encoder-Decoder para transformar) y aplicarlo con pocas líneas de código.

**Próximo encuentro →** APIs de LLMs y Ollama: vas a ver cómo consumir modelos en la nube (Gemini) y cómo correr modelos localmente sin costo de API — y después vas a construir tu primer sistema RAG.